In [ ]:
# eval_champions.ipynb
# Champion selection, SEPARATE from eval.ipynb, in three decoupled steps:
#   1. battery : full downstream battery (TAG k-fold / identity / text-variant /
#                sentiment / recommendation) on the BEST checkpoint of every DONE
#                full-n (n=2000) combo -> per-combo eval_report.json. Resumable.
#   2. extract : scan those reports ONCE -> grid_metrics.json (ALL results, no
#                selection). This is the single source of truth for downstream
#                analysis; later eval stages filter FROM this file.
#   3. select  : read grid_metrics.json -> per-metric top-K INTERSECTION ->
#                champions.json (scores + ranks preserved). Re-tune TOP_K and
#                re-run just this cell -- no report rescan, no GPU.
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG_DIR = "/workspace/stable_query_latent_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG = os.path.join(LOG_DIR, "pipeline_eval_champions.log")
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
SWEEP_YAML = "VICReg_review/sweep/sweep.yaml"

TOP_K = 50            # per-metric cut before intersecting (pool <=96; intersection <= TOP_K)
DIMS_SLICE = None     # e.g. [18, 36] on VM1 and [64, 72] on VM2 to split stage 1
                      # across machines by hand; None = all dims on this machine.

print('repo   :', REPO)
print('out    :', OUT_DIR)
print('log    :', LOG)
print('top-k  :', TOP_K, '| dims slice:', DIMS_SLICE or 'all')


In [ ]:
# FORCE-sync to origin/main (pod repo is a mirror; untracked files untouched).
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD


In [ ]:
# STAGE 1 (battery): per-combo battery on every DONE full-n combo's BEST
# checkpoint. Grid flags derive from sweep.yaml's FULL grid (exclude cleared) so
# masked-but-done cells (e.g. view80) are evaluated too -- but ONLY the full
# train count: n=2000 has the most data, so its per-config comparison is the
# reliable one. per_combo mode skips unfinished combos and caches reports
# (report_is_current), so this cell is safe to re-run any time.
import copy, sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig

_cfg = SweepConfig.load(f'{REPO}/{SWEEP_YAML}')
_full = copy.deepcopy(_cfg)
_full.grid.exclude = []
_g = _full.grid
_counts = [int(n) for n in _g.train_game_counts]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
_dims = [int(d) for d in (DIMS_SLICE or _g.output_dims)]
GAME_COUNTS = str(FULL_N)
VIEWS = ' '.join(f'{float(v):g}' for v in _g.sample_fractions)
DIMS = ' '.join(str(d) for d in _dims)
SCALES = ' '.join(f'{float(s):g}' for s in _g.latent_scales)
EXPANDER_HIDDEN = ' '.join(str(int(h)) for h in _cfg.model.expander_hidden)
ANCHORS = ','.join(str(a) for a in _cfg.data_seed.anchors)
print(f'stage 1 slice: n={GAME_COUNTS}  views=[{VIEWS}]  dims=[{DIMS}]  scales=[{SCALES}]')

!python -u VICReg_review/sweep_cloud.py \
  --h5 game_review_data/embedding_h5.h5 \
  --out-dir {OUT_DIR} \
  --train-game-counts {GAME_COUNTS} \
  --sample-fractions {VIEWS} \
  --output-dims {DIMS} \
  --latent-scales {SCALES} \
  --base-num-latents {_g.base_num_latents} \
  --expander-dim {_cfg.model.expander_dim} \
  --expander-hidden {EXPANDER_HIDDEN} \
  --epochs {_cfg.train.epochs} \
  --batch-size {_cfg.train.batch_size} \
  --skip-train \
  --eval-mode per_combo \
  --calib-mode off \
  --train-game-anchor-appids "{ANCHORS}" \
  --logout-address {LOG}


In [ ]:
# STAGE 2a (extract): scan the full-n eval reports ONCE and write
# grid_metrics.json -- ALL results, NO selection. Single source of truth:
# champion selection, curve filtering, and any later analysis read THIS file
# instead of re-touching per-combo reports.
import copy, json, math, os, socket, subprocess, sys, time
from pathlib import Path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig

_cfg = SweepConfig.load(f'{REPO}/{SWEEP_YAML}')
_full = copy.deepcopy(_cfg)
_full.grid.exclude = []
_counts = [int(c.train_games) for c in _full.iter_combos()]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
live_ids = {c.combo_id for c in _cfg.iter_combos()}
root = Path(REPO) / OUT_DIR

def _mean(vals):
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and math.isnan(v))]
    return sum(vals) / len(vals) if vals else float('nan')

rows, missing = [], []
for c in _full.iter_combos():
    if int(c.train_games) != FULL_N:
        continue
    rp = root / c.combo_id / 'eval_report.json'
    try:
        rep = json.loads(rp.read_text(encoding='utf-8'))
    except Exception:
        missing.append(c.combo_id)
        continue
    tag = (rep.get('tag_probe') or {})
    ident = (rep.get('identity') or {})
    senti = (rep.get('sentiment_probe') or {})
    tv = (rep.get('text_variant_eval') or {})
    rt = tv.get('real_text_tag') or {}
    drops = [(rt.get(v) or {}).get('drop_micro_f1') for v in rt]
    tag_f1 = tag.get('micro_f1')
    r2 = senti.get('r2')
    sel = None
    if tag_f1 is not None and r2 is not None and not math.isnan(float(r2)):
        sel = float(tag_f1) * (1.0 - min(1.0, max(0.0, float(r2))))
    rows.append({
        'combo_id': c.combo_id,
        'arm': c.arm, 'output_dim': c.output_dim, 'num_latents': c.num_latents,
        'view': c.view, 'train_games': c.train_games,
        'masked': c.combo_id not in live_ids,
        'tag_f1': tag_f1,
        'tag_fold_f1_mean': tag.get('fold_micro_f1_mean'),
        'identity_hit_at_1': ident.get('hit_at_1'),
        'identity_mean_rank': ident.get('mean_rank'),
        'identity_pr': ident.get('participation_ratio'),
        'identity_mean_vicreg_cosine': ident.get('mean_vicreg_cosine'),
        'sentiment_r2': r2,
        'variant_drop_mean': _mean(drops),
        'variant_anchor_test_f1': ((tv.get('tag_generalization') or {}).get('anchor_test') or {}).get('micro_f1'),
        'recommendation_pearson': ((rep.get('recommendation_probe') or {}).get('pearson')
                                   if isinstance(rep.get('recommendation_probe'), dict) else None),
        'selectivity': sel,
        'report_path': str(rp),
    })

git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO,
                            capture_output=True, text=True).stdout.strip()
metrics_payload = {
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'git_commit': git_commit,
    'full_n': FULL_N,
    'pool_size': len(rows),
    'missing_reports': missing,
    'rows': rows,
}
metrics_path = root / 'grid_metrics.json'
# unique tmp per writer: a SHARED fixed .tmp lets two concurrent VMs
# interleave write/rename and publish a half-written file
tmp = metrics_path.with_name(f'{metrics_path.name}.tmp.{socket.gethostname()}.{os.getpid()}')
tmp.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=1), encoding='utf-8')
tmp.replace(metrics_path)
print(f'grid_metrics.json: {len(rows)} evaluated full-n combos '
      f'({len(missing)} not evaluated yet) -> {metrics_path}')


In [ ]:
# STAGE 2b (select): read grid_metrics.json ONLY, rank per metric, intersect
# the per-metric top-K sets, write champions.json (scores + ranks preserved).
# Fully decoupled from the reports: tune TOP_K / METRICS and re-run just this
# cell -- instant, no GPU, no report rescan.
import json, math, os, socket, time
from pathlib import Path

metrics_path = Path(REPO) / OUT_DIR / 'grid_metrics.json'
metrics_payload = json.loads(metrics_path.read_text(encoding='utf-8'))
rows = [dict(r) for r in metrics_payload['rows']]

METRICS = {          # name -> (row key, higher_is_better)
    'tag_f1': ('tag_f1', True),
    'identity': ('identity_hit_at_1', True),
    'variant_drop': ('variant_drop_mean', False),
    'selectivity': ('selectivity', True),
}

def _ok(v):
    return v is not None and not (isinstance(v, float) and math.isnan(v))

top_sets, rank_maps = {}, {}
for mname, (key, hi) in METRICS.items():
    scored = [r for r in rows if _ok(r.get(key))]
    scored.sort(key=lambda r: r[key], reverse=hi)
    rank_maps[mname] = {r['combo_id']: i + 1 for i, r in enumerate(scored)}
    top_sets[mname] = [r['combo_id'] for r in scored[:TOP_K]]

champion_ids = set(r['combo_id'] for r in rows)
for ids in top_sets.values():
    champion_ids &= set(ids)
for r in rows:
    r['ranks'] = {m: rank_maps[m].get(r['combo_id']) for m in METRICS}
    r['rank_sum'] = sum(v for v in r['ranks'].values() if v is not None)
champions = sorted((r for r in rows if r['combo_id'] in champion_ids),
                   key=lambda r: r['rank_sum'])

payload = {
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'source_metrics': {'path': str(metrics_path),
                       'created_at': metrics_payload.get('created_at'),
                       'git_commit': metrics_payload.get('git_commit'),
                       'pool_size': metrics_payload.get('pool_size')},
    'full_n': metrics_payload.get('full_n'),
    'top_k': TOP_K,
    'selection': 'intersection of per-metric top-K over all done full-n combos '
                 '(both arms in one pool; selectivity normalises the arm difference)',
    'metrics': {m: {'key': k, 'higher_is_better': hi} for m, (k, hi) in METRICS.items()},
    'top_sets': top_sets,
    'champions': champions,
}
out_path = Path(REPO) / OUT_DIR / 'champions.json'
tmp = out_path.with_name(f'{out_path.name}.tmp.{socket.gethostname()}.{os.getpid()}')
tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding='utf-8')
tmp.replace(out_path)

print(f'pool: {len(rows)} combos from grid_metrics.json '
      f'({len(metrics_payload.get("missing_reports", []))} not evaluated yet)')
for m in METRICS:
    print(f'  {m:13} ranked {len(rank_maps[m])} combos, top-{TOP_K} kept')
print(f'champions (intersection): {len(champions)} -> {out_path}')
print()
print(f"{'combo_id':44} {'tagF1':>6} {'id@1':>5} {'vdrop':>6} {'select':>6} {'ranks':>16}")
for r in champions:
    print(f"{r['combo_id']:44} {r['tag_f1'] or float('nan'):6.3f} "
          f"{r['identity_hit_at_1'] or float('nan'):5.2f} "
          f"{r['variant_drop_mean']:6.3f} {r['selectivity'] or float('nan'):6.3f} "
          f"{str([r['ranks'][m] for m in METRICS]):>16}")
